# Kepler.gl / PyDeck Census Maps

This notebook focuses on high-performance 3D visualization of US Census data. 

> **Note:** Due to compatibility issues between `keplergl` and Python 3.13 (specifically `pyarrow` build failures), we are using **PyDeck** (`deck.gl` for Python). PyDeck is the underlying technology that powers Kepler.gl and offers the same high-performance WebGL rendering capabilities.

## Setup

In [1]:
import pandas as pd
import geopandas as gpd
from census import Census
import pydeck as pdk

# Set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)

## 1. Fetch & Prepare Data
We reuse the data extraction logic from `modern_census_maps.ipynb`.

In [2]:
# REPLACE WITH YOUR API KEY
CENSUS_API_KEY = "942e0a44c121ca03ced84b727df9b004f1f1367d"
c = Census(CENSUS_API_KEY)

# Variables: Median Income (B19013_001E), Population (B01003_001E)
VARIABLES = {
    "B19013_001E": "Median_Income",
    "B01003_001E": "Population"
}

try:
    # Fetch data for all states (Year 2021)
    # We pass the list of variable codes as a tuple
    census_data = c.acs5.get(("NAME", *VARIABLES.keys()), {'for': 'state:*', 'year': 2021})
    
    if not census_data:
        raise ValueError("API returned empty data.")
        
    # Convert to DataFrame
    df_census = pd.DataFrame(census_data)
    
    # Rename columns using the mapping
    df_census = df_census.rename(columns=VARIABLES)
    df_census = df_census.rename(columns={"state": "STATEFP"})
    
    # Convert numeric columns
    for col in VARIABLES.values():
        df_census[col] = pd.to_numeric(df_census[col])
    
    print(f"Successfully fetched data for {len(df_census)} states.")
    print(df_census.head())
    
except Exception as e:
    print(f"Error fetching data: {e}")


Successfully fetched data for 52 states.
         NAME  Median_Income  Population STATEFP
0     Alabama        62027.0   5054253.0      01
1      Alaska        89336.0    733971.0      02
2     Arizona        76872.0   7268175.0      04
3    Arkansas        58773.0   3032651.0      05
4  California        96334.0  39242785.0      06


In [3]:
# Load US States Geometry
url = "https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_state_20m.zip"
try:
    gdf_states = gpd.read_file(url)
    
    # Merge with Census Data
    gdf_states = gdf_states.merge(df_census, on="STATEFP", how="left")
    
    # Rename 'NAME_x' back to 'NAME' if collision occurred
    if 'NAME_x' in gdf_states.columns:
        gdf_states = gdf_states.rename(columns={'NAME_x': 'NAME'})
        
    # Filter out missing data
    gdf_states = gdf_states.dropna(subset=['Median_Income'])
    
    # Ensure EPSG:4326 for PyDeck
    gdf_states = gdf_states.to_crs(epsg=4326)
    
    print("Geometry loaded and merged successfully.")
except Exception as e:
    print(f"Error loading geometry: {e}")

Geometry loaded and merged successfully.


In [4]:
# DEBUG: Filter to just one state (TX) to verify code logic and avoid compute timeouts
print('Debugging: Filtering dataset to Texas (TX) only.')
gdf_states = gdf_states[gdf_states['STUSPS'] == 'TX'].copy()
print(f'Filtered GDF size: {len(gdf_states)}')


Debugging: Filtering dataset to Texas (TX) only.
Filtered GDF size: 1


In [5]:
# Generate Synthetic Population Points (Optimized)
import numpy as np
from shapely.geometry import Point

def generate_random_points_in_polygon(polygon, num_points):
    # 1. OPTION A: Fast Geopandas sampling (if available)
    try:
        # Check if sample_points exists on GeoSeries
        gs = gpd.GeoSeries([polygon])
        if hasattr(gs, 'sample_points'):
            sampled = gs.sample_points(num_points)
            # Explode in case of MultiPoint outputs
            coords = sampled.explode(index_parts=False)
            return [Point(p.x, p.y) for p in coords]
    except Exception as e:
        # Fallback if specific version issues occur
        pass

    # 2. OPTION B: Safer Rejection Sampling
    points = []
    min_x, min_y, max_x, max_y = polygon.bounds
    attempts = 0
    # Max attempts to prevent infinite loops on weird geometries
    max_attempts = num_points * 100 + 1000 
    
    while len(points) < num_points and attempts < max_attempts:
        attempts += 1
        rand_x = np.random.uniform(min_x, max_x)
        rand_y = np.random.uniform(min_y, max_y)
        p = Point(rand_x, rand_y)
        if polygon.contains(p):
            points.append(p)
            
    if len(points) < num_points:
        print(f"Warning: Could only generate {len(points)}/{num_points} points for a geometry.")
        
    return points

points_data = []
SCALE = 100000  # 1 dot = 100k people

print(f"Generating synthetic points (Scale: 1 dot = {SCALE:,} people)...")

for idx, row in gdf_states.iterrows():
    pop = row.get('Population', 0)
    if pd.isna(pop) or pop <= 0:
        continue
    
    n_points = int(pop / SCALE)
    # Safety Cap
    if n_points > 2000: n_points = 2000
    
    if n_points > 0:
        try:
            pts = generate_random_points_in_polygon(row.geometry, n_points)
            for p in pts:
                points_data.append({
                    "lng": p.x,
                    "lat": p.y,
                    "state": row["NAME"]
                })
        except Exception as e:
            print(f"Skipping {row['NAME']}: {e}")

df_points = pd.DataFrame(points_data)

# Ensure types are JSON serializable for PyDeck
if not df_points.empty:
    df_points['lng'] = df_points['lng'].astype(float)
    df_points['lat'] = df_points['lat'].astype(float)
    df_points['state'] = df_points['state'].astype(str)
print(f"Generated {len(df_points)} points representing US population density.")


Generating synthetic points (Scale: 1 dot = 100,000 people)...
Generated 296 points representing US population density.


## 2. PyDeck 3D Map
Visualizing Median Household Income as extruded polygons. 
- **Height**: Represents Income
- **Color**: Represents Income (Red to Green scale)

In [6]:
# Normalize income for color scale (Simple scaling for demo)
min_inc = gdf_states['Median_Income'].min()
max_inc = gdf_states['Median_Income'].max()

def get_color(income):
    import pandas as pd
    import numpy as np
    if pd.isna(income):
        return [200, 200, 200, 100]
    
    denom = max_inc - min_inc
    if denom == 0 or pd.isna(denom):
        norm = 0
    else:
        norm = (income - min_inc) / denom
    
    if pd.isna(norm):
        return [200, 200, 200, 100]
        
    # Clamp
    norm = max(0.0, min(1.0, norm))
    
    return [int(255 * (1 - norm)), int(255 * norm), 100, 200]  # RGBA
gdf_states['color'] = gdf_states['Median_Income'].apply(get_color)

# view_state for US
view_state = pdk.ViewState(
    latitude=37.0902,
    longitude=-95.7129,
    zoom=3,
    pitch=45,
    bearing=0
)

layer = pdk.Layer(
    "GeoJsonLayer",
    gdf_states,
    pickable=True,
    stroked=True,
    filled=True,
    extruded=True,
    wireframe=True,
    get_fill_color="color",
    get_line_color=[255, 255, 255],
    get_elevation=f"{'Median_Income'} * 0.1", # Scale factor for height
    auto_highlight=True,
)

deck = pdk.Deck(
    layers=[layer],
    initial_view_state=view_state,
    tooltip={"text": "{NAME}\nMedian Income: ${Median_Income}"},
    map_style="road"
)

# Render in Notebook
deck.to_html("census_3d_map.html")
deck.show()

In [7]:
# 3D Hexagon Layer Map (Hexbin Map)
# This plot aggregates data points (state centroids) into hexagonal areas and extrudes them based on Mean Income.

# Calculate centroids for point-based aggregation
# Note: ensuring we are working with the projected or geographic CRS as needed. 
# PyDeck expects lon/lat in EPSG:4326, which gdf_states should already be in from previous cells.
gdf_states['lon'] = gdf_states.geometry.centroid.x
gdf_states['lat'] = gdf_states.geometry.centroid.y

# Define the Hexagon Layer
hex_layer = pdk.Layer(
    "HexagonLayer",
    gdf_states,
    get_position=['lon', 'lat'],
    radius=150000,         # Radius in meters (adjust based on desired granularity)
    elevation_scale=100,
    elevation_range=[0, 3000],
    extruded=True,
    pickable=True,
    get_elevation_weight='Median_Income',  # Uses the variable defined earlier, e.g., "Median_Income"
    elevation_aggregation="MEAN",        # Aggregates using the mean of the values in the bin
    auto_highlight=True,
)

# View State centered on US
view_state = pdk.ViewState(
    latitude=37.0902,
    longitude=-95.7129,
    zoom=3,
    pitch=45,
    bearing=0
)

# Render Deck
deck_hex = pdk.Deck(
    layers=[hex_layer],
    initial_view_state=view_state,
    tooltip={
        "html": "<b>Mean Income:</b> {elevationValue}",
        "style": {"color": "white"}
    },
    map_style="dark",
)

# Render in Notebook and save HTML
deck_hex.to_html("census_hex_map.html")
deck_hex.show()

/var/folders/ss/l5wyf1v92rl0qj7z7zx8zzhw0000gn/T/ipykernel_15739/3081116877.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_states['lon'] = gdf_states.geometry.centroid.x
/var/folders/ss/l5wyf1v92rl0qj7z7zx8zzhw0000gn/T/ipykernel_15739/3081116877.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_states['lat'] = gdf_states.geometry.centroid.y


In [8]:
# New Hampshire County Level 3D Hexagon Map
# More precise visualization for NH counties

NH_FIPS = '33'

try:
    # 1. Fetch County Data for NH
    print("Fetching county data for New Hampshire...")
    nh_census = c.acs5.get(("NAME", "B19013_001E"), {'for': 'county:*', 'in': f'state:{NH_FIPS}', 'year': 2021})
    
    if not nh_census:
        raise ValueError("API returned empty data for NH counties.")
        
    df_nh = pd.DataFrame(nh_census)
    df_nh = df_nh.rename(columns={"B19013_001E": 'Median_Income', "state": "STATEFP", "county": "COUNTYFP"})
    df_nh['Median_Income'] = pd.to_numeric(df_nh['Median_Income'])
    
    # 2. Geometry: Load US Counties and Filter for NH
    # Note: Using 20m resolution for consistency, but 5m or 500k might be better for zooming.
    # Since we used 20m for states, we'll use 20m for counties unless precise borders are needed.
    # Given 'more precise' request, let's try to stick to what works first.
    county_url = "https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_county_20m.zip"
    print("Loading US County geometry...")
    gdf_counties = gpd.read_file(county_url)
    
    # Filter for NH
    gdf_nh = gdf_counties[gdf_counties['STATEFP'] == NH_FIPS].copy()
    
    # 3. Merge
    gdf_nh = gdf_nh.merge(df_nh, on=["STATEFP", "COUNTYFP"], how="left")
    gdf_nh = gdf_nh.dropna(subset=['Median_Income'])
    gdf_nh = gdf_nh.to_crs(epsg=4326)
    
    # 4. Calculate Centroids for Hex Points
    gdf_nh['lon'] = gdf_nh.geometry.centroid.x
    gdf_nh['lat'] = gdf_nh.geometry.centroid.y
    
    # 5. Hexagon Layer (Higher Precision)
    hex_layer_nh = pdk.Layer(
        "HexagonLayer",
        gdf_nh,
        get_position=['lon', 'lat'],
        radius=5000,           # 5km radius for finer granularity
        elevation_scale=100,
        elevation_range=[0, 3000],
        extruded=True,
        pickable=True,
        get_elevation_weight='Median_Income',
        elevation_aggregation="MEAN",
        auto_highlight=True,
        coverage=0.9           # Slightly separated hexagons
    )
    
    # View State centered on NH
    view_state_nh = pdk.ViewState(
        latitude=43.7939,     # Approx NH center
        longitude=-71.5724,
        zoom=7,
        pitch=45,
        bearing=0
    )
    
    deck_nh = pdk.Deck(
        layers=[hex_layer_nh],
        initial_view_state=view_state_nh,
        tooltip={
            "html": "<b>Mean Income:</b> {elevationValue}",
            "style": {"color": "white"}
        },
        map_style="dark",
    )
    
    deck_nh.to_html("nh_county_hex_map.html")
    deck_nh.show()
    print("NH County Map generated successfully!")
    
except Exception as e:
    print(f"Error generating NH map: {e}")

Fetching county data for New Hampshire...
Loading US County geometry...
NH County Map generated successfully!


/var/folders/ss/l5wyf1v92rl0qj7z7zx8zzhw0000gn/T/ipykernel_15739/2548808101.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_nh['lon'] = gdf_nh.geometry.centroid.x
/var/folders/ss/l5wyf1v92rl0qj7z7zx8zzhw0000gn/T/ipykernel_15739/2548808101.py:36: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_nh['lat'] = gdf_nh.geometry.centroid.y


## 3. Population Density Hexbin Map
This map aggregates the synthetic population points into hexagons. The height and color of each hexagon represent the density of points (people) in that area.


## 4. San Francisco Building Footprints (3D)
Visualizing building heights in San Francisco using 3D extrusion.

In [9]:
# Load SF Building Footprints directly from SF Open Data
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape

# SODA API Endpoint (Limit to 5000 buildings for performance)
DATA_URL = "https://data.sfgov.org/resource/ynuv-fyni.json?$limit=3000"

try:
    print("Fetching SF Building Footprints from DataSF...")
    df_sf = pd.read_json(DATA_URL)
    
    # Convert JSON geometry to Shapely geometry
    # The 'shape' column contains GeoJSON-like dicts
    if 'shape' in df_sf.columns:
        df_sf['geometry'] = df_sf['shape'].apply(lambda x: shape(x) if isinstance(x, dict) else None)
        gdf_sf = gpd.GeoDataFrame(df_sf, geometry='geometry')
        gdf_sf.crs = "EPSG:4326" # SODA API is usually 4326
        
        # Handle missing heights
        gdf_sf['hgt_median_m'] = gdf_sf['hgt_median_m'].fillna(10) # Default to 10m
        
        print(f"Loaded {len(gdf_sf)} buildings.")

        # Define Layer
        layer_sf = pdk.Layer(
            "GeoJsonLayer",
            gdf_sf,
            opacity=0.8,
            stroked=False,
            filled=True,
            extruded=True,
            wireframe=False,
            get_elevation="hgt_median_m",
            get_fill_color=[255, 100, 100],
            get_line_color=[255, 255, 255],
            pickable=True,
            auto_highlight=True,
        )

        # View State (San Francisco)
        view_state_sf = pdk.ViewState(
            latitude=37.7749,
            longitude=-122.4194,
            zoom=13,
            pitch=60,
            bearing=30
        )

        deck_sf = pdk.Deck(
            layers=[layer_sf],
            initial_view_state=view_state_sf,
            tooltip={"text": "Height: {hgt_median_m}m"},
            map_style="dark"
        )

        deck_sf.to_html("sf_buildings_real.html")
        deck_sf.show()
        print("SF Buildings Map generated successfully.")
    else:
        print("Error: 'shape' column not found in dataset.")

except Exception as e:
    print(f"Error generating SF map: {e}")


Fetching SF Building Footprints from DataSF...
Loaded 3000 buildings.
SF Buildings Map generated successfully.
